# Vietnam GRASP Replication Driver

This notebook is the lightweight driver for the fresh-data Vietnam replication. It deliberately ignores Fleur's historical `.npy` and pickle files. The workflow is:

1. Use the current PISA pipeline to build fresh population, source, candidate, and distance-matrix files.
2. Convert the PISA parquets into sparse max-cover `.npz` instances.
3. Compare greedy construction, sparse first-swap local search, randomized GRASP, and fast path relinking.
4. Read the batch result tables and inspect selected candidate sites.

The notebook is intentionally source-only. It does not embed large local results.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path(r"C:/local/GIT/Public-Infrastructure-Service-Access")
VIETNAM_ROOT = REPO_ROOT / "Research-Sandbox" / "Parvathy_PhD" / "Vietnam"
OUTPUT_ROOT = Path(r"C:/local/Parvathy/Vietnam")
SCRIPTS = VIETNAM_ROOT / "scripts"

sys.path.insert(0, str(SCRIPTS))

VIETNAM_ROOT, OUTPUT_ROOT

## 1. Fresh PISA Pipeline Command

Run this from a terminal when the fresh network-distance data are missing. The full national run writes outputs outside git under `C:/local/Parvathy/Vietnam`.

In [ ]:
pipeline_command = r'''
cd C:\local\GIT\Public-Infrastructure-Service-Access\Research-Sandbox\Parvathy_PhD\Vietnam
py scripts\run_vietnam_fresh.py vietnam ^
  --sources table candidates ^
  --destinations population ^
  --source-table "C:\Users\joaqu\OneDrive - UvA\share\Vietnam\stroke-facs-100-en.xlsx" ^
  --source-lon-column longitude ^
  --source-lat-column latitude ^
  --source-id-column TT ^
  --aggregate-factor 10 ^
  --candidate-grid-spacing-m 10000 ^
  --candidate-max-snap-dist-m 5000 ^
  --max-total-dist 150000 ^
  --matrix-output-mode split ^
  --matrix-shape sparse ^
  --diagnose-connectivity true ^
  --log-file C:\local\Parvathy\Vietnam\logs\fresh_table_candidate_10kmgrid_150km_no_map.log
'''
print(pipeline_command)

## 2. Build Max-Cover Instances

The replication uses three service thresholds: 20 km, 50 km, and 100 km. Existing stroke facilities form the baseline; candidates add incremental coverage.

In [ ]:
outputs_dir = OUTPUT_ROOT / "fresh_downloads" / "vietnam_data" / "outputs"
optimization_dir = OUTPUT_ROOT / "optimization"
run_tag = "candidates_spacing_10000_maxsnap_5000"

instance_commands = []
for threshold_km in [20, 50, 100]:
    out = optimization_dir / f"vietnam_10kmgrid_{threshold_km}km_threshold.npz"
    instance_commands.append(
        f"py scripts\\build_pisa_instance.py --outputs-dir {outputs_dir} "
        f"--run-tag-marker {run_tag} --threshold-m {threshold_km * 1000} "
        f"--output-npz {out}"
    )

print("\n".join(instance_commands))

## 3. Load One Instance

The cells below illustrate the different heuristic families on one fresh instance. Change `threshold_km` and `budget` to explore other cases.

In [ ]:
import pandas as pd

from run_vietnam_fleur_style_analysis import load_instance, objective_to_population
from vietnam_grasp_heuristics import budgeted_construct, improve_local_search, run_grasp

threshold_km = 50
budget = 20
instance_path = optimization_dir / f"vietnam_10kmgrid_{threshold_km}km_threshold.npz"

if not instance_path.exists():
    raise FileNotFoundError(f"Build the instance first: {instance_path}")

loaded = load_instance(instance_path)
loaded.path.name, loaded.instance.n_facilities, loaded.instance.n_households, loaded.baseline, loaded.total_population

## 4. Greedy Construction

In [ ]:
greedy = budgeted_construct(loaded.instance, budget, constructor="greedy")
greedy_incremental = objective_to_population(loaded, greedy.objective)
greedy_total = loaded.baseline + greedy_incremental

{
    "method": "greedy",
    "budget": budget,
    "incremental_population": greedy_incremental,
    "total_covered_population": greedy_total,
    "coverage_percent": 100 * greedy_total / loaded.total_population,
    "seconds": greedy.total_time,
}

## 5. Sparse First-Swap Local Search

`first_sparse` preserves the first-improving 1-for-1 swap idea but uses a sparse candidate-neighborhood index to avoid the heavy Python candidate-collection loop.

In [ ]:
local = improve_local_search(loaded.instance, greedy, local_search="first_sparse")
local_incremental = objective_to_population(loaded, local.objective)
local_total = loaded.baseline + local_incremental

{
    "method": "greedy_first_sparse",
    "budget": budget,
    "incremental_population": local_incremental,
    "total_covered_population": local_total,
    "coverage_percent": 100 * local_total / loaded.total_population,
    "seconds": greedy.total_time + local.total_time,
    "accepted_local_moves": max(0, len(local.objectives) - 1),
}

## 6. Randomized GRASP With Fast Path Relinking

This is a short stochastic illustration. Production runs should increase `max_iterations`, `time_limit_seconds`, and repeats.

In [ ]:
grasp, records = run_grasp(
    loaded.instance,
    budget=budget,
    constructor="randomized",
    rcl_size=25,
    local_search="first_sparse",
    path_relinking=True,
    path_relinking_method="fast",
    max_iterations=3,
    time_limit_seconds=60,
    seed=42,
)
grasp_incremental = objective_to_population(loaded, grasp.objective)
grasp_total = loaded.baseline + grasp_incremental

pd.DataFrame([
    {
        "method": "greedy",
        "incremental_population": greedy_incremental,
        "total_covered_population": greedy_total,
        "seconds": greedy.total_time,
    },
    {
        "method": "greedy_first_sparse",
        "incremental_population": local_incremental,
        "total_covered_population": local_total,
        "seconds": greedy.total_time + local.total_time,
    },
    {
        "method": "grasp_first_sparse_fast_pr",
        "incremental_population": grasp_incremental,
        "total_covered_population": grasp_total,
        "seconds": grasp.total_time,
    },
])

In [ ]:
pd.DataFrame([record.__dict__ for record in records])

## 7. Batch Fleur-Style Analysis

The batch runner writes the coverage tables, selected candidate table, traces, and plots used in the replication notes.

In [ ]:
batch_command = r'''
py scripts\run_vietnam_fleur_style_analysis.py ^
  --instances ^
    C:\local\Parvathy\Vietnam\optimization\vietnam_10kmgrid_20km_threshold.npz ^
    C:\local\Parvathy\Vietnam\optimization\vietnam_10kmgrid_50km_threshold.npz ^
    C:\local\Parvathy\Vietnam\optimization\vietnam_10kmgrid_100km_threshold.npz ^
  --budgets 20 40 60 80 100 200 ^
  --local-search-budgets 20 40 60 80 100 200 ^
  --randomized-budgets 20 40 60 80 ^
  --randomized-repeats 3 ^
  --grasp-max-iterations 5 ^
  --grasp-time-limit-seconds 120 ^
  --local-search first_sparse ^
  --path-relinking-method fast ^
  --output-dir C:\local\Parvathy\Vietnam\fleur_style_fast_replication
'''
print(batch_command)

## 8. Read Batch Results

In [ ]:
results_dir = OUTPUT_ROOT / "fleur_style_fast_replication"
summary_path = results_dir / "coverage_summary_by_budget.csv"

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.head())
    best = (
        summary.sort_values(
            ["threshold_km", "budget", "total_covered_population", "seconds"],
            ascending=[True, True, False, True],
        )
        .groupby(["threshold_km", "budget"], as_index=False)
        .head(1)
    )
    display(best[["threshold_km", "budget", "method", "total_covered_population", "coverage_percent_total_population", "seconds"]])
else:
    print(f"No batch summary yet: {summary_path}")

## 9. Dense-Grid Screening Notes

For 5 km and 1 km candidate grids, full national network-distance matrices can become too large. The repository includes a grid-only builder and a straight-line spatial screening runner. These are not substitutes for final road-network-distance PISA results; they are fast screening tools for dense grids.

In [ ]:
dense_grid_commands = r'''
py scripts\build_candidate_grid_only.py --candidate-grid-spacing-m 1000 ^
  --summary-json C:\local\Parvathy\Vietnam\dense_grid_straightline_analysis\candidate_grid_1km_summary.json

py scripts\run_dense_grid_straightline_analysis.py ^
  --candidate-grid C:\local\Parvathy\Vietnam\fresh_downloads\vietnam_data\cache\vnm_candidate_sites_spacing_5000m_water_allowed_include_boundary_epsg_3405.pkl ^
  --grid-spacing-m 5000 ^
  --thresholds-km 20 50 100 ^
  --budgets 20 80 200 ^
  --local-search-budgets 20 80 200 ^
  --randomized-budgets 20 ^
  --output-dir C:\local\Parvathy\Vietnam\dense_grid_straightline_analysis\grid5km
'''
print(dense_grid_commands)